# V2.4 · 成交量因子稳健性与收益归因（Jupyter 研究记录）

**目的：让数据运行、图表、结果解释与复现信息留在同一个 Notebook。**

本文件基于仓库 `fix/v2.4-factor-audit` 的 V2.4 实现；**只调用已有研究引擎，不修改 V1、V2.1、V2.2、V2.3 或原始行情。**

研究任务：比较动态成交量选股、初始固定名单月度等权、初始固定名单买入持有和全股票池等权；分析个股真实 P&L 贡献、排除某股后的重跑结果，以及成交量单位变化的敏感性。

> 注意：`demo` 使用**人工数据**；真实数据也有固定股票池幸存者偏差、理想化收盘成交、假设费用和历史数据口径问题。所有历史时间分段均是事后分析，不是新的独立样本外认证。

**操作顺序：** 环境检查 → （可选）测试 → 修改配置 → 运行并自动保存快照 → 表格与可视化 → 填写研究结论 → 用 Jupyter 菜单 File → Save Notebook 保存本次解释与图表。

## 1. 环境与项目定位

使用和 V2.4 相同的 Python 虚拟环境作为 Kernel。若下面显示的 Python 路径不是项目中的 `.venv/bin/python`，先在 Jupyter 右上角切换 Kernel。

In [ ]:
from pathlib import Path
import sys
import json
import platform
import subprocess
from datetime import datetime
from uuid import uuid4

import pandas as pd
import numpy as np
from IPython.display import display, Markdown
import matplotlib.pyplot as plt

HERE = Path.cwd().resolve()
ROOT = next(
    (p for p in [HERE, *HERE.parents] if (p / "v2_4" / "factor_audit_v2_4.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("请从完整 GitHub 仓库目录打开 Notebook，不能只复制 .ipynb 文件")
V24 = ROOT / "v2_4"
sys.path.insert(0, str(V24))

from factor_audit_v2_4 import (
    Data, demo_data, read_data, download, run_audit, save_run,
    compare_previous, verify_engines,
)

print("Python:", sys.version.split()[0])
print("Kernel:", sys.executable)
print("Repository:", ROOT)
print("V2.4:", V24)
verify_engines()
assert (ROOT / "v2_2" / "backtest_v2_2.py").is_file()
print("✔ V2.1 / V2.2 引擎源代码校验通过")

## 2. （可选）在 Notebook 内执行自动化测试

测试信息会直接留在单元格输出里，不需要在 Terminal 查看。修改核心代码后建议运行一次。默认运行 V2.4 与 V2.1 的原有测试。

In [ ]:
RUN_TESTS = False  # 初次使用建议改为 True；以后纯分析可保持 False
if RUN_TESTS:
    proc = subprocess.run(
        [sys.executable, "-m", "pytest", "-q", "tests", "../v2_1/tests"],
        cwd=V24, capture_output=True, text=True,
    )
    print(proc.stdout)
    if proc.returncode:
        print(proc.stderr)
        raise RuntimeError(f"测试失败，退出码={proc.returncode}")
else:
    print("本次跳过测试；在需要时把 RUN_TESTS 设为 True")

## 3. 配置本次研究（先从 demo 开始）

- `MODE = "demo"`：人工数据，检查程序和图表。
- `MODE = "download"`：下载新一份 2018–2024 历史行情快照；需联网并安装 yfinance。
- `MODE = "csv"`：读取已保存的 `close.csv`、`volume.csv`，也可包含 `source_close.csv`、`splits.csv`。**重跑同一份快照时优先用此模式。**

每次运行自动创建**唯一的新输出目录**，不覆盖旧结果。项目中的 `outputs_jupyter/` 会被 Git 忽略。

In [ ]:
MODE = "demo"  # 改成 "download" 或 "csv" 时才分析真实数据
START, END = "2018-01-01", "2025-01-01"  # END 不包含当天
N_STOCKS = 10
COST_BPS = 10.0       # 每笔买入、卖出名义金额的假设费率
LEAVE_ONE_OUT = True  # 对每只股票逐一排除并重新运行，真实数据需要较长时间
CSV_FOLDER = Path.home() / "Downloads" / "QR_V24_Jupyter" / "v2_4" / "my_saved_snapshot"
REFERENCE_V23 = (Path.home() / "Downloads" / "QR_V23_separate" /
                 "v2_3" / "outputs_v23_real_2018_2024")  # 可不存在
INITIAL_CASH = 1_000_000.0

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid4().hex[:6]
RUN_DIR = V24 / "outputs_jupyter" / RUN_ID
assert MODE in {"demo", "download", "csv"}
assert not RUN_DIR.exists()
print("模式:", MODE)
print("本次专属输出目录:", RUN_DIR)

## 4. 执行分析并保存所有输入快照和结果

本单元格调用 V2.4 中已测试的核算引擎；**不修改交易策略，不自动删除股票，不下真实订单**。运行失败时请保留报错并检查输入，不要用事后删除缺失数据的方式“修好”结果。

In [ ]:
verify_engines()
if MODE == "demo":
    close, volume = demo_data()
    data = Data(close, volume, close.copy(), volume * 0.0)
    data_label = "SYNTHETIC DEMO — NOT MARKET EVIDENCE"
elif MODE == "csv":
    data = read_data(CSV_FOLDER)
    data_label = "CSV SNAPSHOT — CHECK DATA PROVENANCE"
else:
    data = download(START, END)
    data_label = "NEW YAHOO SNAPSHOT — NOT EXACT V2.3 REPLAY"

result = run_audit(
    data, cost_bps=COST_BPS, n=N_STOCKS,
    initial=INITIAL_CASH, leave_one_out=LEAVE_ONE_OUT,
)

meta = {
    "source": data_label,
    "cost_bps": COST_BPS, "n_stocks": N_STOCKS,
    "lookback": 20, "ic_horizon": 20, "leave_one_out": LEAVE_ONE_OUT,
    "first_date": str(data.close.index[0]),
    "last_date": str(data.close.index[-1]),
    "tickers": list(data.close.columns),
    "initial_cash": INITIAL_CASH,
    "execution": "month-end signal / next session close trade",
    "limitations": [
        "Retrospective analysis, not untouched holdout",
        "Fixed universe and survivorship bias",
        "Provider price/volume not point-in-time certified",
        "Idealized next-close fills and assumed proportional cost",
        "True historical float turnover NOT RUN",
    ],
}
if MODE == "download":
    import yfinance
    meta.update(requested_start=START, requested_end_exclusive=END,
                yfinance_version=yfinance.__version__)
if MODE != "demo" and REFERENCE_V23.is_dir():
    meta["reference_v23"] = compare_previous(
        REFERENCE_V23, result["accounts"]["dynamic_volume"]
    )

if RUN_DIR.exists():
    raise FileExistsError(f"输出目录已存在，不会覆盖：{RUN_DIR}")
save_run(RUN_DIR, data, result, meta)
print("✔ 已保存研究快照与全部结果")
print("来源:", data_label)
print("日期范围:", meta["first_date"], "至", meta["last_date"])
print("输出位置:", RUN_DIR)
print("V2.3 对照:", meta.get("reference_v23", "未执行；需旧结果文件及真实数据"))
print("逐股票损益与账户损益最大核对残差:", result["attribution_residual"])

## 5. 研究表格：动态选股和静态对照

同一首次信号确定入场日。比较策略 CAGR、Sharpe、IR 及最大回撤。若是 demo，**所有数字仅说明代码能够运行，不代表股票市场真实表现**。

In [ ]:
full = result["comparison"].query("period == 'full'").copy()
display(full[[
    "strategy", "n_days", "cagr", "sharpe_rf_zero",
    "ir_vs_universe", "max_drawdown", "fees",
]].sort_values("strategy"))

fig, ax = plt.subplots(figsize=(9, 4))
plot_frame = full.set_index("strategy")["cagr"].mul(100).sort_values()
plot_frame.plot.barh(ax=ax)
ax.set_xlabel("Full-period CAGR (%)")
ax.set_title("Dynamic vs Static Controls — " + data_label)
fig.tight_layout()
fig.savefig(RUN_DIR / "comparison_cagr.png", dpi=150)
plt.show()

with open(RUN_DIR / "frozen_selection.json", encoding="utf-8") as f:
    frozen = json.load(f)
print("首次固定名单选择信息:", frozen)

## 6. 收益归因：到底是哪只股票贡献了损益？

`contribution_to_total_return` 是**净损益／初始资金**；它能跨股票求和得到组合累计收益率，但不是单股 CAGR。排除股票重跑属于另外一种反事实实验，不能与原账户损益贡献混为一谈。

In [ ]:
attrib = result["attribution"].sort_values("net_pnl", ascending=False)
display(attrib)
print("累计股票净损益:", round(float(attrib["net_pnl"].sum()), 4))
print("累计账户净值变化:", round(
    float(result["accounts"]["dynamic_volume"].daily.equity_after_trade.iloc[-1] - INITIAL_CASH), 4
))

fig, ax = plt.subplots(figsize=(9, 6))
attrib["contribution_to_total_return"].mul(100).sort_values().plot.barh(ax=ax)
ax.set_xlabel("Contribution to cumulative portfolio return (percentage points)")
ax.set_title("Per-stock realized/mark-to-market P&L net of assumed fees")
fig.tight_layout()
fig.savefig(RUN_DIR / "attribution.png", dpi=150)
plt.show()

## 7. 稳健性检查：排除股票、成交量单位变化、拆股事件

如果去掉 NVDA 后策略表现变化，**并不等于 NVDA 在原组合中贡献了同等金额**；因为其他股票重新入选、权重和交易费用都会改变。

真实流通股换手率仍为 `NOT_RUN`，因为目前没有可靠的历史流通股数及发布时间数据。

In [ ]:
print("逐一排除股票：")
if len(result["leave_one_out"]):
    display(result["leave_one_out"].sort_values("cagr_change_vs_original").head(15))
else:
    print("本次禁用了 leave-one-out")

print("成交量单位敏感性（单股票成交量整体乘2）：")
display(result["unit_stress"].sort_values("changed_selections", ascending=False).head(15))

print("数据定义与运行边界：")
display(result["definitions"])
print("拆股事件检查：")
display(result["split_audit"].head(20))

## 8. 研究记录（请运行后填写并保存 Notebook）

**本次数据：** Demo / Yahoo 新快照 / 已校验快照，日期：____。

**假设 H1：** 动态成交量选股是否优于首次固定名单、同频率再平衡？

**观察：** ____________________________

**假设 H2：** 是否少数股票主导了累计净损益？

**观察：** ____________________________

**假设 H3：** 对成交股数尺度与股票池排除是否敏感？

**观察：** ____________________________

**尚未解决：** 固定股票池幸存者偏差；历史流通股数；企业行动的点时可得性；价差、市场冲击和成交容量；未接触过的独立样本外数据。

**下次实验建议：** ____________________________

### 留档方法

1. Jupyter 菜单 **File → Save Notebook**。代码、输出图表和这份文字记录都会留在 `.ipynb` 中。
2. 原始行情快照、`RUN_INFO.json` 与分析 CSV 位于上方打印的 `RUN_DIR`，不会覆盖旧次运行。
3. 若要分享分析，使用 **File → Export Notebook As → HTML**（如界面提供此选项）；原 `.ipynb` 永远保留作为可编辑研究记录。
